# 6주차 ① 옵티마이저와 학습률 스케줄러 — 실습 1~2

**목표**: 같은 모델·같은 시드에서 SGD / Momentum / Adam 을 비교해 **선택 기준**을 세우고,
스케줄러를 붙여 **학습률이 시간에 따라 변하는 것**을 곡선으로 확인한다.

> **실행 전 확인** — 커널 `Python (dl2026)`. 5주차의 `data/` 를 그대로 씁니다.

> **오늘은 새 데이터를 쓰지 않습니다.** 5주차와 같은 FashionMNIST 입니다.
> 데이터를 바꾸면 *"같은 문제에서 무엇이 달라지는가"* 라는 비교 구도가 흐려집니다.
> 비교 실습은 전부 **epoch 5 고정** — "충분히 학습"이 아니라 **"차이를 본다"** 가 목적입니다.

| 단계 | 무엇을 해결했나 | 한 줄 |
|---|---|---|
| **SGD** | (출발점) | `p -= lr * g` — 4주차에 손으로 쓴 그것 |
| **+ Momentum** | 지그재그 | **관성**을 준다. 계속 같은 방향이면 점점 빨라진다 |
| **Adam** | 축마다 다른 기울기 크기 | **파라미터마다 학습률을 다르게** 자동 조절 |
| **AdamW** | Adam 의 가중치 감쇠가 어긋남 | 가중치 감쇠를 **기울기에서 분리**. 9·12주차 기본 |

> **함정 ★ — lr 의 자릿수가 다릅니다.** SGD 는 `0.1` 대, Adam 은 `1e-3` 대가 기본입니다.
> **Adam 에 0.1 을 주면 대개 발산합니다.** 옵티마이저를 바꾸면 **lr 도 함께 바꿔야** 합니다.

## 실습 1 — SGD vs Momentum vs Adam ★

> **공정한 비교의 조건**: 모델·시드·데이터·epoch 를 전부 같게 두고 **옵티마이저만** 바꿉니다.
> 6주차 과제의 채점 기준이 정확히 이것입니다.

In [ ]:
# 셀 1 — 공통 준비
import torch, torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print("사용할 장치 :", device)

ROOT = "data"
tf = transforms.Compose([transforms.ToTensor(),
                         transforms.Normalize((0.2860,), (0.3530,))])

train_set = datasets.FashionMNIST(ROOT, train=True,  download=True, transform=tf)
test_set  = datasets.FashionMNIST(ROOT, train=False, download=True, transform=tf)
train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False)

def make_model():
    torch.manual_seed(0)                     # ★ 항상 같은 초기값에서 출발
    return nn.Sequential(nn.Flatten(),
                         nn.Linear(784, 256), nn.ReLU(),
                         nn.Linear(256, 128), nn.ReLU(),
                         nn.Linear(128, 10)).to(device)

In [ ]:
# 셀 2 — 학습 함수 (5주차 루프 그대로)
def train(opt_name, epochs=5, sched=None):
    model = make_model()
    lossfn = nn.CrossEntropyLoss()
    opt = {
        "SGD":      torch.optim.SGD(model.parameters(), lr=0.1),
        "Momentum": torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9),
        "Adam":     torch.optim.Adam(model.parameters(), lr=1e-3),
    }[opt_name]
    scheduler = sched(opt) if sched else None

    hist, lrs = [], []
    for _ in range(epochs):
        model.train(); run, n = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = lossfn(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
            run += loss.item() * xb.size(0); n += xb.size(0)
        lrs.append(opt.param_groups[0]["lr"])      # ★ 지금의 학습률
        if scheduler: scheduler.step()             # ★ epoch 끝에 한 번
        hist.append(run / n)
    return model, hist, lrs

def accuracy(model):
    model.eval(); c = t = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            c += (model(xb).argmax(1) == yb).sum().item(); t += yb.size(0)
    return c / t

In [ ]:
# 셀 3 — 3종 비교
results, accs = {}, {}
for name in ["SGD", "Momentum", "Adam"]:
    m, h, _ = train(name)
    accs[name] = accuracy(m)
    results[name] = h
    print(f"{name:9s} | 마지막 loss {h[-1]:.4f} | 테스트 정확도 {accs[name]*100:5.2f}%")

for name, h in results.items():
    plt.plot(range(1, len(h)+1), h, marker="o", label=name)
plt.xlabel("epoch"); plt.ylabel("평균 loss"); plt.legend()
plt.title("옵티마이저 3종 (같은 모델·같은 시드·5 epoch)"); plt.show()

> **관찰 포인트 ★**: 초반 수렴은 대개 **Adam 이 가장 빠릅니다.** 그런데 5 epoch 뒤 최종 정확도는
> **Momentum 이 비슷하거나 더 높은 경우**가 흔합니다.
> *"빨리 내려가는 것"* 과 *"끝까지 잘 가는 것"* 은 다른 이야기입니다.
> **Adam 이 항상 이기지는 않습니다.**

| 상황 | 권장 |
|---|---|
| 처음 시도 · 빠른 실험 | **Adam / AdamW** — 튜닝 없이도 잘 붙는다 |
| 이미지 분류에서 최고 성능 | **SGD + Momentum** — 잘 맞추면 Adam 을 이기는 경우가 많다 |
| 사전학습 모델 미세조정 | **AdamW** — 9·12주차에서 쓴다 |

In [ ]:
# 셀 4 — 옵티마이저를 바꿀 때 lr 도 바꿔야 하는 이유 (일부러 발산시킨다)
torch.manual_seed(0)
bad = make_model()
bad_opt = torch.optim.Adam(bad.parameters(), lr=0.1)      # ★ Adam 에 SGD 용 lr
lossfn = nn.CrossEntropyLoss()

bad.train()
for i, (xb, yb) in enumerate(train_loader):
    xb, yb = xb.to(device), yb.to(device)
    loss = lossfn(bad(xb), yb)
    bad_opt.zero_grad(); loss.backward(); bad_opt.step()
    if i % 100 == 0:
        print(f"iter {i:4d} | loss {loss.item():.4f}")
    if i == 400: break

print(f"\n테스트 정확도 : {accuracy(bad)*100:.2f}%   ← 찍기 수준이면 발산한 것")

> **결과 해석 ★**: 손실이 안 줄거나 튑니다. **`lr=1e-3` 이었으면 잘 됐을 코드**입니다.
> *"옵티마이저를 바꿀 때는 lr 도 같이 바꾼다"* — 이 한 줄이 오늘의 실용적 결론입니다.

### 기록할 것 (과제 비교표의 뼈대)

| 기록할 것 | 왜 |
|---|---|
| 옵티마이저 이름, **학습률** | lr 이 다르면 공정한 비교가 아니다 |
| 시드 | 다시 돌렸을 때 같은 결과가 나와야 한다 (3교시) |
| epoch, batch size | 조건이 같아야 비교가 성립 |
| 최종 손실, 테스트 정확도 | 결과 |

In [ ]:
# 셀 5 — 비교표를 지금 만들어 둔다 (과제에 그대로 쓴다)
print(f"{'옵티마이저':10s} {'lr':>8s} {'epoch':>6s} {'batch':>6s} "
      f"{'seed':>5s} {'최종loss':>9s} {'정확도':>8s}")
print("-" * 62)
for name, lr in [("SGD", 0.1), ("Momentum", 0.1), ("Adam", 1e-3)]:
    print(f"{name:10s} {lr:>8} {5:>6} {128:>6} {0:>5} "
          f"{results[name][-1]:>9.4f} {accs[name]*100:>7.2f}%")

## 실습 2 — 스케줄러 적용 + 학습률 곡선

```
   초반: 멀리 있으니 크게 가야 한다        lr 크게
   후반: 바닥 근처에서 크게 가면 튄다      lr 작게

        loss
         │＼
         │  ＼___              ← 초반: 성큼성큼
         │      ＼__
         │         ‾●‿●‿●     ← 후반: lr 이 그대로면 바닥에서 진동한다
         └────────────────── epoch
```

| 스케줄러 | 동작 | 쓰임 |
|---|---|---|
| **`StepLR`** | 정해진 epoch마다 lr 에 `gamma` 를 곱한다 | 가장 단순 |
| **`CosineAnnealingLR`** | 코사인 곡선으로 부드럽게 0 쪽으로 | **요즘 기본** ★ 9·12주차 |
| `ReduceLROnPlateau` | 검증 손실이 안 줄면 낮춘다 | 이름만 알아 두기 |

> **함정 ★★**: `scheduler.step()` 을 **배치 안쪽에 두면** lr 이 469배 빨리 줄어 학습이 멈춥니다.
> **epoch 루프 끝**에 둡니다. 매년 나오는 실수입니다.
> `optimizer.step()`(파라미터) 과 `scheduler.step()`(학습률) 은 **완전히 다른 일**을 합니다.

In [ ]:
# 셀 6 — 스케줄러 유무 비교
from torch.optim.lr_scheduler import StepLR, CosineAnnealingLR

configs = {
    "스케줄러 없음": None,
    "StepLR":       lambda o: StepLR(o, step_size=2, gamma=0.5),
    "Cosine":       lambda o: CosineAnnealingLR(o, T_max=5),
}

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name, sched in configs.items():
    m, h, lrs = train("Momentum", epochs=5, sched=sched)
    ax[0].plot(range(1, 6), h,   marker="o", label=name)
    ax[1].plot(range(1, 6), lrs, marker="o", label=name)
    print(f"{name:12s} | 마지막 loss {h[-1]:.4f} | 정확도 {accuracy(m)*100:5.2f}%")

ax[0].set_title("손실"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].set_title("학습률의 변화"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.show()

> **관찰 포인트 ★**: 오른쪽 그림에서 **학습률이 실제로 줄어드는 모양**을 보세요.
> `StepLR` 은 계단, `Cosine` 은 부드러운 곡선입니다.
> 왼쪽 손실 차이는 **5 epoch 로는 작을 수 있습니다** — 스케줄러의 효과는 **긴 학습에서** 커집니다.

> 여기서 중요한 것은 *"스케줄러를 쓰면 무조건 좋아진다"* 가 아닙니다.
> **학습률이 시간에 따라 변한다는 개념**과, 그것을 **곡선으로 확인하는 습관**입니다.

> **막히면**:
> | 증상 | 원인 |
> |---|---|
> | 학습률이 안 변한다 | `scheduler.step()` 을 안 불렀다 |
> | 2 epoch 만에 lr 이 0 에 가까워진다 | `scheduler.step()` 이 배치 루프 안에 있다 ★ |
> | 손실이 발산 | Adam 에 `lr=0.1` 을 줬다 |

In [ ]:
# 셀 7 (참고) — 학습 없이 스케줄러 모양만 미리 본다
for label, make in [("StepLR(step=2, γ=0.5)", lambda o: StepLR(o, 2, 0.5)),
                    ("Cosine(T_max=20)",      lambda o: CosineAnnealingLR(o, 20))]:
    o = torch.optim.SGD([torch.zeros(1, requires_grad=True)], lr=0.1)
    s = make(o)
    curve = []
    for _ in range(20):
        curve.append(o.param_groups[0]["lr"])
        o.step(); s.step()
    plt.plot(curve, marker=".", label=label)

plt.xlabel("epoch"); plt.ylabel("learning rate"); plt.legend()
plt.title("스케줄러 모양만 20 epoch 미리 보기"); plt.show()

> **포인트**: 학습을 돌리기 전에 **스케줄러 모양만 먼저 그려 보는 것**은
> 좋은 습관입니다. `T_max` 나 `step_size` 를 잘못 줘서 lr 이 너무 빨리 0 이 되는 사고를 막습니다.

---

### 이 노트북 체크리스트

- [ ] SGD → Momentum → Adam → AdamW 가 **각각 무엇을 해결했는지** 말할 수 있다 ★
- [ ] Adam 의 기본 lr 이 SGD 와 자릿수가 다른 것을 안다
- [ ] 같은 시드·같은 모델로 옵티마이저 3종을 비교했다
- [ ] **Adam 이 항상 이기지는 않는다**는 것을 결과로 확인했다
- [ ] Adam 에 `lr=0.1` 을 줘서 발산하는 것을 직접 봤다
- [ ] 스케줄러를 적용하고 **학습률 곡선**을 그렸다
- [ ] `scheduler.step()` 의 위치를 안다 ★★
- [ ] 비교 실험에 기록해야 할 조건 4가지를 안다